# Omani Speaker Fine-tuning (parallel, one model per speaker)

Loads a **pretrained Tacotron2 checkpoint**, then fine-tunes a **separate copy** of the
model for each Omani speaker in parallel using Python `multiprocessing`.

| Setting | Default |
|---|---|
| Pretrained checkpoint | set `PRETRAINED_CHECKPOINT` below |
| Speakers to train | `SPEAKERS` — subset of `[B, C, D, E]` |
| Epochs per speaker | `FINETUNE_EPOCHS` |
| Checkpoints kept | milestone every `SAVE_EVERY` epochs + final |
| Output dir | `checkpoints_omani/<speaker>/` |

> **GPU note:** each worker claims the same GPU if only one is available.
> Set `DEVICE` to a dict like `{"B": "cuda:0", "C": "cuda:1"}` for multi-GPU.

## 0. Install requirements

In [ ]:
! pip install -q -r ../requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.1 MB/s eta 0:00:00


## 0a. Download Omani dataset from Google Drive

Downloads two files into `dataset_new_omani/`:
- **Audio zip** extracted to `clean_flac/clean_flac/*.flac`
- **Transcriptions** saved as `transcriptions.xlsx`

Each file is skipped if it already exists on disk.

In [ ]:
import subprocess, sys, zipfile, shutil
from pathlib import Path

# ── Google Drive ID ───────────────────────────────────────────────────────────
DATASET_ZIP_FILE_ID = "1nFMp2NmH4hkuz9Qc7NRjq6OSj7vMAGzQ"  # contains clean_flac/*.flac + transcriptions.xlsx

# ── Resolve project root ──────────────────────────────────────────────────────
_root = Path.cwd().resolve()
if _root.name == "tacotron2":
    _root = _root.parent
for _ in range(4):
    if (_root / "commons" / "dataset.py").exists():
        break
    _root = _root.parent

# Target layout:
#   dataset_new_omani/
#   ├── transcriptions.xlsx
#   └── clean_flac/
#       └── *.flac
DATASET_DEST = _root / "dataset_new_omani"
AUDIO_DIR    = DATASET_DEST / "clean_flac"
XLSX_DEST    = DATASET_DEST / "transcriptions.xlsx"

# ── Ensure gdown is available ─────────────────────────────────────────────────
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--root-user-action=ignore", "gdown"])
    import gdown

# ── Skip if already extracted ─────────────────────────────────────────────────
if AUDIO_DIR.exists() and any(AUDIO_DIR.glob("*.flac")) and XLSX_DEST.exists():
    print(f"Dataset already present ({len(list(AUDIO_DIR.glob('*.flac')))} flac files) — skipping.")
else:
    zip_path = DATASET_DEST / "dataset.zip"
    DATASET_DEST.mkdir(parents=True, exist_ok=True)
    print("Downloading dataset zip ...")
    gdown.download(id=DATASET_ZIP_FILE_ID, output=str(zip_path), quiet=False, fuzzy=True)
    print(f"Extracting {zip_path.name} ...")
    AUDIO_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            filename = Path(member).name
            if not filename:  # skip directory entries
                continue
            if member.lower().endswith(".flac"):
                with zf.open(member) as src, open(AUDIO_DIR / filename, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            elif member.lower().endswith(".xlsx"):
                with zf.open(member) as src, open(XLSX_DEST, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    zip_path.unlink()
    print(f"Extracted {len(list(AUDIO_DIR.glob('*.flac')))} .flac files to {AUDIO_DIR}")
    print(f"Extracted transcriptions.xlsx to {XLSX_DEST}")

print("\nDataset ready at:", DATASET_DEST)


## 0b. Download processed Omani dataset from Google Drive folder

In [ ]:
import subprocess, sys
from pathlib import Path

# ── Google Drive folder ID ────────────────────────────────────────────────────
PROCESSED_FOLDER_ID = "1rSEAMHGYLxhqITSmFzduAX3TtLgb-OGK"

# ── Destination: project_root/ (folder will be created as dataset_new_omani_processed/) ──
PROCESSED_DEST = _root / "dataset_new_omani_processed"

# ── Ensure gdown is available ─────────────────────────────────────────────────
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--root-user-action=ignore", "gdown"])
    import gdown

# ── Skip if already downloaded ────────────────────────────────────────────────
if PROCESSED_DEST.exists() and any(PROCESSED_DEST.iterdir()):
    print(f"Processed dataset already present at {PROCESSED_DEST} — skipping.")
else:
    print(f"Downloading processed dataset folder to {_root} ...")
    gdown.download_folder(
        id=PROCESSED_FOLDER_ID,
        output=str(_root),
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
    )
    print("Download complete. Contents:")
    for f in sorted(PROCESSED_DEST.iterdir()):
        print(" ", f.name)

print("Processed dataset ready at:", PROCESSED_DEST)


## 0. Environment setup

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tacotron2":
    ROOT = ROOT.parent
for _ in range(4):
    if (ROOT / "commons" / "dataset.py").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

## 1. Configuration — edit here

In [ ]:
from pathlib import Path

# ── Pretrained checkpoint ──────────────────────────────────────────────────────
PRETRAINED_CHECKPOINT = str(ROOT / "checkpoints" / "tacotron2_last.pth")

# ── Dataset ────────────────────────────────────────────────────────────────────
DATASET_ROOT = str(ROOT / "dataset_new_omani")

# ── Speaker label (single Omani speaker) ──────────────────────────────────────
SPEAKERS = ["omani"]

# ── Training hyperparameters ───────────────────────────────────────────────────
FINETUNE_EPOCHS = 300
BATCH_SIZE      = 16
LEARNING_RATE   = 1e-4
GRAD_CLIP       = 1.0

# ── Guided attention loss ──────────────────────────────────────────────────────
GUIDED_ATTN_WEIGHT       = 5.0
GUIDED_ATTN_SIGMA        = 0.2
GUIDED_ATTN_ANNEAL_START = 100
GUIDED_ATTN_ANNEAL_END   = 200

# ── Validation split ───────────────────────────────────────────────────────────
VAL_SPLIT      = 0.1
VAL_SPLIT_SEED = 42

# ── Checkpointing ──────────────────────────────────────────────────────────────
CKPT_BASE_DIR  = str(ROOT / "checkpoints_omani")
SAVE_EVERY     = 10
KEEP_LAST_ONLY = True

# ── Parallelism & device ───────────────────────────────────────────────────────
MAX_PARALLEL = 1
DEVICE       = "auto"

print("Configuration:")
print(f"  PRETRAINED_CHECKPOINT    : {PRETRAINED_CHECKPOINT}")
print(f"  DATASET_ROOT             : {DATASET_ROOT}")
print(f"  SPEAKERS                 : {SPEAKERS}")
print(f"  FINETUNE_EPOCHS          : {FINETUNE_EPOCHS}")
print(f"  BATCH_SIZE               : {BATCH_SIZE}")
print(f"  LEARNING_RATE            : {LEARNING_RATE}")
print(f"  GUIDED_ATTN_WEIGHT       : {GUIDED_ATTN_WEIGHT}")
print(f"  GUIDED_ATTN_SIGMA        : {GUIDED_ATTN_SIGMA}")
print(f"  GUIDED_ATTN_ANNEAL_START : {GUIDED_ATTN_ANNEAL_START}")
print(f"  GUIDED_ATTN_ANNEAL_END   : {GUIDED_ATTN_ANNEAL_END}")
print(f"  CKPT_BASE_DIR            : {CKPT_BASE_DIR}")
print(f"  SAVE_EVERY               : {SAVE_EVERY}")
print(f"  MAX_PARALLEL             : {MAX_PARALLEL}")
print(f"  DEVICE                   : {DEVICE}")


In [ ]:
import torch
import torch.nn.functional as F
import os, shutil, sys, traceback
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader, random_split


def _resolve_device(device_cfg, speaker):
    if isinstance(device_cfg, dict):
        return torch.device(device_cfg.get(speaker, "cpu"))
    if device_cfg == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_cfg)


def _show_alignment_grid(attn_np, mel_true_np, mel_pred_np, epoch, speaker, ga_w, save_path):
    """
    Render the 3-panel alignment figure.
    - Always saves to save_path (PNG on disk).
    - If IPython is available (notebook kernel, not subprocess), also displays inline.
    attn_np     : (T_dec, T_enc)
    mel_true_np : (T_dec, n_mels)
    mel_pred_np : (T_dec, n_mels)
    """
    # Use inline backend when IPython is present, Agg otherwise
    try:
        from IPython import get_ipython
        _in_notebook = get_ipython() is not None
    except ImportError:
        _in_notebook = False

    if _in_notebook:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    else:
        matplotlib.use("Agg")

    import matplotlib.pyplot as plt  # re-import after backend switch

    fig, axes = plt.subplots(3, 1, figsize=(10, 9))

    im0 = axes[0].imshow(attn_np, aspect="auto", origin="lower", interpolation="nearest")
    axes[0].set_title(f"Speaker {speaker} — epoch {epoch}  (ga_w={ga_w:.3f})")
    axes[0].set_xlabel("Encoder step")
    axes[0].set_ylabel("Decoder step")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(mel_true_np.T, aspect="auto", origin="lower", interpolation="nearest")
    axes[1].set_title("Ground-truth mel")
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Mel bin")
    fig.colorbar(im1, ax=axes[1])

    vmin, vmax = mel_true_np.min(), mel_true_np.max()
    im2 = axes[2].imshow(mel_pred_np.T, aspect="auto", origin="lower",
                         interpolation="nearest", vmin=vmin, vmax=vmax)
    axes[2].set_title("Predicted mel (post-net)")
    axes[2].set_xlabel("Frame")
    axes[2].set_ylabel("Mel bin")
    fig.colorbar(im2, ax=axes[2])

    plt.tight_layout()
    fig.savefig(save_path, dpi=100)

    if _in_notebook:
        from IPython.display import display
        display(fig)

    plt.close(fig)

    # Restore Agg so subsequent non-display saves work correctly
    matplotlib.use("Agg")


def _guided_attention_loss(attn_weights, enc_mask, dec_mask, sigma):
    """
    Soft diagonal attention penalty (Tachibana et al. 2018).

    attn_weights : (B, T_dec, T_enc)
    enc_mask     : (B, T_enc) bool — True = padding
    dec_mask     : (B, T_dec) bool — True = padding
    sigma        : float — diagonal bandwidth as fraction of sequence length
    """
    B, T_dec, T_enc = attn_weights.shape
    device = attn_weights.device

    S = (~enc_mask.bool()).sum(dim=1).float() if enc_mask is not None else torch.full((B,), T_enc, dtype=torch.float, device=device)
    T = (~dec_mask.bool()).sum(dim=1).float() if dec_mask is not None else torch.full((B,), T_dec, dtype=torch.float, device=device)

    t_norm = (torch.arange(T_dec, device=device).float().unsqueeze(0) / T.unsqueeze(1)).unsqueeze(2)
    s_norm = (torch.arange(T_enc, device=device).float().unsqueeze(0) / S.unsqueeze(1)).unsqueeze(1)

    W = 1.0 - torch.exp(-((t_norm - s_norm) ** 2) / (2 * sigma ** 2))
    loss = attn_weights * W

    if dec_mask is not None:
        loss = loss.masked_fill(dec_mask.bool().unsqueeze(2), 0.0)
    if enc_mask is not None:
        loss = loss.masked_fill(enc_mask.bool().unsqueeze(1), 0.0)

    if dec_mask is not None and enc_mask is not None:
        valid = (~dec_mask.bool()).float().unsqueeze(2) * (~enc_mask.bool()).float().unsqueeze(1)
        n_valid = valid.sum().clamp(min=1.0)
    else:
        n_valid = float(B * T_dec * T_enc)

    return loss.sum() / n_valid


def _guided_attn_weight_at_epoch(epoch, peak, anneal_start, anneal_end):
    """Linear decay from peak to 0 between anneal_start and anneal_end."""
    if epoch <= anneal_start:
        return peak
    if epoch >= anneal_end:
        return 0.0
    return peak * (1.0 - (epoch - anneal_start) / max(anneal_end - anneal_start, 1))


def finetune_speaker(speaker, cfg_dict):
    """Entry point for each subprocess. cfg_dict carries all hyperparams."""
    root = cfg_dict["root"]
    if root not in sys.path:
        sys.path.insert(0, root)

    from commons.dataset     import NewOmaniTTSDataset, TTSCollator
    from commons.hyperparams import Tacotron2Config
    from tacotron2.model     import Tacotron2
    from tacotron2.tokenizer import Tokenizer

    device = _resolve_device(cfg_dict["device"], speaker)
    print(f"[{speaker}] Starting on {device}")

    config = Tacotron2Config()
    tok = Tokenizer()
    config.num_chars    = tok.vocab_size
    config.pad_token_id = tok.pad_token_id

    full_ds = NewOmaniTTSDataset(
        cfg_dict["dataset_root"],
        sample_rate=config.sample_rate,
        n_fft=config.n_fft,
        window_size=config.win_length,
        hop_size=config.hop_length,
        fmin=config.fmin,
        fmax=config.fmax,
        num_mels=config.n_mels,
        min_db=config.min_db,
        max_scaled_abs=config.max_scaled_abs,
    )
    n_val   = max(1, int(len(full_ds) * cfg_dict["val_split"]))
    n_train = len(full_ds) - n_val
    gen = torch.Generator().manual_seed(cfg_dict["val_seed"])
    train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=gen)

    collate_fn   = TTSCollator()
    train_loader = DataLoader(
        train_ds, batch_size=cfg_dict["batch_size"],
        shuffle=True, collate_fn=collate_fn, num_workers=0,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg_dict["batch_size"],
        shuffle=False, collate_fn=collate_fn, num_workers=0,
    )
    print(f"[{speaker}] train={n_train}  val={n_val}")

    model = Tacotron2(config).to(device)
    ck = torch.load(
        cfg_dict["pretrained_ckpt"], map_location=device, weights_only=False
    )
    model.load_state_dict(ck["model_state_dict"], strict=True)
    base_ep = ck.get("epoch", "?")
    print(f"[{speaker}] Loaded pretrained weights (base epoch {base_ep})")

    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg_dict["lr"], eps=config.eps
    )

    ckpt_dir = Path(cfg_dict["ckpt_base_dir"]) / speaker
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    align_dir = ckpt_dir / "alignments"
    align_dir.mkdir(exist_ok=True)

    # Fixed single validation sample — same utterance every checkpoint for consistent comparison.
    probe_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)
    probe_batch = next(iter(probe_loader))

    train_losses, val_losses = [], []
    last_ckpt_path = None
    epochs          = cfg_dict["epochs"]
    grad_clip       = cfg_dict["grad_clip"]
    keep_last       = cfg_dict["keep_last_only"]
    save_every      = cfg_dict["save_every"]
    ga_peak         = cfg_dict.get("guided_attn_weight", 0.0)
    ga_sigma        = cfg_dict.get("guided_attn_sigma", 0.2)
    ga_anneal_start = cfg_dict.get("guided_attn_anneal_start", epochs)
    ga_anneal_end   = cfg_dict.get("guided_attn_anneal_end", epochs)

    for epoch in range(1, epochs + 1):
        ga_w = _guided_attn_weight_at_epoch(epoch, ga_peak, ga_anneal_start, ga_anneal_end)

        # --- train ---
        model.train()
        ep_loss = 0.0
        for batch in train_loader:
            text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = batch
            text_pad = text_pad.to(device)
            mel_pad  = mel_pad.to(device)
            gate_pad = gate_pad.to(device)
            enc_mask = enc_mask.to(device)
            dec_mask = dec_mask.to(device)
            optimizer.zero_grad(set_to_none=True)
            mel_out, mel_post, stop_tok, attn = model(
                text_pad, in_lens, mel_pad, enc_mask, dec_mask
            )
            loss = (
                F.mse_loss(mel_out, mel_pad)
                + F.mse_loss(mel_post, mel_pad)
                + F.binary_cross_entropy_with_logits(
                    stop_tok.reshape(-1, 1), gate_pad.reshape(-1, 1)
                )
            )
            if ga_w > 0.0:
                loss = loss + ga_w * _guided_attention_loss(attn, enc_mask, dec_mask, ga_sigma)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            ep_loss += loss.item()
        avg_train = ep_loss / max(len(train_loader), 1)
        train_losses.append(avg_train)

        # --- validate ---
        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = batch
                text_pad = text_pad.to(device)
                mel_pad  = mel_pad.to(device)
                gate_pad = gate_pad.to(device)
                enc_mask = enc_mask.to(device)
                dec_mask = dec_mask.to(device)
                mel_out, mel_post, stop_tok, _ = model(
                    text_pad, in_lens, mel_pad, enc_mask, dec_mask
                )
                v_loss += (
                    F.mse_loss(mel_out, mel_pad)
                    + F.mse_loss(mel_post, mel_pad)
                    + F.binary_cross_entropy_with_logits(
                        stop_tok.reshape(-1, 1), gate_pad.reshape(-1, 1)
                    )
                ).item()
        avg_val = v_loss / max(len(val_loader), 1)
        val_losses.append(avg_val)
        print(f"[{speaker}] Epoch {epoch:3d}/{epochs} -- train {avg_train:.4f}  val {avg_val:.4f}  ga_w {ga_w:.3f}")

        # --- alignment: probe fixed sample, save PNG and display inline ---
        if epoch % save_every == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                p_text, p_lens, p_mel, _, p_enc_mask, p_dec_mask = probe_batch
                p_text     = p_text.to(device)
                p_mel      = p_mel.to(device)
                p_enc_mask = p_enc_mask.to(device)
                p_dec_mask = p_dec_mask.to(device)
                _, p_mel_post, _, p_attn = model(p_text, p_lens, p_mel, p_enc_mask, p_dec_mask)

            attn_np     = p_attn[0].cpu().float().numpy()      # (T_dec, T_enc)
            mel_true_np = p_mel[0].cpu().float().numpy()       # (T_dec, n_mels)
            mel_pred_np = p_mel_post[0].cpu().float().numpy()  # (T_dec, n_mels)

            plot_path = align_dir / f"epoch_{epoch:04d}.png"
            _show_alignment_grid(attn_np, mel_true_np, mel_pred_np,
                                 epoch, speaker, ga_w, plot_path)
            print(f"[{speaker}]   Alignment saved → {plot_path.name}")

        # --- checkpoint ---
        is_milestone = (epoch % save_every == 0) or (epoch == epochs)
        ckpt_name = (
            f"speaker_{speaker}_epoch_{epoch:04d}.pth"
            if is_milestone
            else "speaker_{}_last.pth".format(speaker)
        )
        new_path = ckpt_dir / ckpt_name
        torch.save({
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch"               : epoch,
            "epochs_total"        : epochs,
            "speaker"             : speaker,
            "train_losses"        : train_losses,
            "val_losses"          : val_losses,
            "pretrained_from"     : cfg_dict["pretrained_ckpt"],
            "config"              : config,
        }, new_path)

        if keep_last and last_ckpt_path and last_ckpt_path.exists():
            stem = last_ckpt_path.stem
            if "_epoch_" in stem:
                prev_ep = int(stem.split("_epoch_")[-1])
                if prev_ep % save_every != 0:
                    last_ckpt_path.unlink(missing_ok=True)
            else:
                last_ckpt_path.unlink(missing_ok=True)

        last_ckpt_path = new_path
        print(f"[{speaker}]   Saved {new_path.name}")

    final_alias = ckpt_dir / f"speaker_{speaker}_last.pth"
    if last_ckpt_path and last_ckpt_path != final_alias:
        shutil.copy2(last_ckpt_path, final_alias)

    print(f"[{speaker}] Done. Final checkpoint: {final_alias}")
    return speaker, train_losses, val_losses


print("Worker function defined.")

## 2. Validate checkpoint and dataset

In [ ]:
from commons.dataset import NewOmaniTTSDataset
from commons.hyperparams import Tacotron2Config
from tacotron2.tokenizer import Tokenizer

assert os.path.isfile(PRETRAINED_CHECKPOINT), (
    f"Checkpoint not found: {PRETRAINED_CHECKPOINT}\n"
    "Train a base model with tacotron2_training.ipynb first."
)

config = Tacotron2Config()
tok = Tokenizer()
config.num_chars    = tok.vocab_size
config.pad_token_id = tok.pad_token_id

ds = NewOmaniTTSDataset(
    DATASET_ROOT,
    sample_rate=config.sample_rate, n_fft=config.n_fft,
    window_size=config.win_length, hop_size=config.hop_length,
    fmin=config.fmin, fmax=config.fmax, num_mels=config.n_mels,
    min_db=config.min_db, max_scaled_abs=config.max_scaled_abs,
)
n_val   = max(1, int(len(ds) * VAL_SPLIT))
n_train = len(ds) - n_val
print(f"Dataset: {len(ds)} total  |  train={n_train}  val={n_val}")


## 3. Fine-tuning worker (runs in a subprocess per speaker)

In [ ]:
import torch
import torch.nn.functional as F
import os, shutil, sys, traceback
from pathlib import Path
from torch.utils.data import DataLoader, random_split


def _resolve_device(device_cfg, speaker):
    if isinstance(device_cfg, dict):
        return torch.device(device_cfg.get(speaker, "cpu"))
    if device_cfg == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_cfg)


def _guided_attention_loss(attn_weights, enc_mask, dec_mask, sigma):
    B, T_dec, T_enc = attn_weights.shape
    device = attn_weights.device

    S = (~enc_mask.bool()).sum(dim=1).float() if enc_mask is not None else torch.full((B,), T_enc, dtype=torch.float, device=device)
    T = (~dec_mask.bool()).sum(dim=1).float() if dec_mask is not None else torch.full((B,), T_dec, dtype=torch.float, device=device)

    t_norm = (torch.arange(T_dec, device=device).float().unsqueeze(0) / T.unsqueeze(1)).unsqueeze(2)
    s_norm = (torch.arange(T_enc, device=device).float().unsqueeze(0) / S.unsqueeze(1)).unsqueeze(1)

    W = 1.0 - torch.exp(-((t_norm - s_norm) ** 2) / (2 * sigma ** 2))
    loss = attn_weights * W

    if dec_mask is not None:
        loss = loss.masked_fill(dec_mask.bool().unsqueeze(2), 0.0)
    if enc_mask is not None:
        loss = loss.masked_fill(enc_mask.bool().unsqueeze(1), 0.0)

    if dec_mask is not None and enc_mask is not None:
        valid = (~dec_mask.bool()).float().unsqueeze(2) * (~enc_mask.bool()).float().unsqueeze(1)
        n_valid = valid.sum().clamp(min=1.0)
    else:
        n_valid = float(B * T_dec * T_enc)

    return loss.sum() / n_valid


def _guided_attn_weight_at_epoch(epoch, peak, anneal_start, anneal_end):
    if epoch <= anneal_start:
        return peak
    if epoch >= anneal_end:
        return 0.0
    return peak * (1.0 - (epoch - anneal_start) / max(anneal_end - anneal_start, 1))


def finetune_speaker(speaker, cfg_dict):
    """Entry point for each subprocess. cfg_dict carries all hyperparams."""
    root = cfg_dict["root"]
    if root not in sys.path:
        sys.path.insert(0, root)

    from commons.dataset     import NewOmaniTTSDataset, TTSCollator
    from commons.hyperparams import Tacotron2Config
    from tacotron2.model     import Tacotron2
    from tacotron2.tokenizer import Tokenizer

    device = _resolve_device(cfg_dict["device"], speaker)
    print(f"[{speaker}] Starting on {device}")

    config = Tacotron2Config()
    tok = Tokenizer()
    config.num_chars    = tok.vocab_size
    config.pad_token_id = tok.pad_token_id

    full_ds = NewOmaniTTSDataset(
        cfg_dict["dataset_root"],
        sample_rate=config.sample_rate,
        n_fft=config.n_fft,
        window_size=config.win_length,
        hop_size=config.hop_length,
        fmin=config.fmin,
        fmax=config.fmax,
        num_mels=config.n_mels,
        min_db=config.min_db,
        max_scaled_abs=config.max_scaled_abs,
    )
    n_val   = max(1, int(len(full_ds) * cfg_dict["val_split"]))
    n_train = len(full_ds) - n_val
    gen = torch.Generator().manual_seed(cfg_dict["val_seed"])
    train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=gen)

    collate_fn   = TTSCollator()
    train_loader = DataLoader(
        train_ds, batch_size=cfg_dict["batch_size"],
        shuffle=True, collate_fn=collate_fn, num_workers=0,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg_dict["batch_size"],
        shuffle=False, collate_fn=collate_fn, num_workers=0,
    )
    print(f"[{speaker}] train={n_train}  val={n_val}")

    model = Tacotron2(config).to(device)
    ck = torch.load(
        cfg_dict["pretrained_ckpt"], map_location=device, weights_only=False
    )
    model.load_state_dict(ck["model_state_dict"], strict=True)
    base_ep = ck.get("epoch", "?")
    print(f"[{speaker}] Loaded pretrained weights (base epoch {base_ep})")

    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg_dict["lr"], eps=config.eps
    )

    ckpt_dir = Path(cfg_dict["ckpt_base_dir"]) / speaker
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    align_dir = ckpt_dir / "alignments"
    align_dir.mkdir(exist_ok=True)

    # Fixed single validation sample — same sample every epoch for consistent comparison.
    probe_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)
    probe_batch = next(iter(probe_loader))

    train_losses, val_losses = [], []
    last_ckpt_path = None
    epochs          = cfg_dict["epochs"]
    grad_clip       = cfg_dict["grad_clip"]
    keep_last       = cfg_dict["keep_last_only"]
    save_every      = cfg_dict["save_every"]
    ga_peak         = cfg_dict.get("guided_attn_weight", 0.0)
    ga_sigma        = cfg_dict.get("guided_attn_sigma", 0.2)
    ga_anneal_start = cfg_dict.get("guided_attn_anneal_start", epochs)
    ga_anneal_end   = cfg_dict.get("guided_attn_anneal_end", epochs)

    for epoch in range(1, epochs + 1):
        ga_w = _guided_attn_weight_at_epoch(epoch, ga_peak, ga_anneal_start, ga_anneal_end)

        model.train()
        ep_loss = 0.0
        for batch in train_loader:
            text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = batch
            text_pad = text_pad.to(device)
            mel_pad  = mel_pad.to(device)
            gate_pad = gate_pad.to(device)
            enc_mask = enc_mask.to(device)
            dec_mask = dec_mask.to(device)
            optimizer.zero_grad(set_to_none=True)
            mel_out, mel_post, stop_tok, attn = model(
                text_pad, in_lens, mel_pad, enc_mask, dec_mask
            )
            loss = (
                F.mse_loss(mel_out, mel_pad)
                + F.mse_loss(mel_post, mel_pad)
                + F.binary_cross_entropy_with_logits(
                    stop_tok.reshape(-1, 1), gate_pad.reshape(-1, 1)
                )
            )
            if ga_w > 0.0:
                loss = loss + ga_w * _guided_attention_loss(attn, enc_mask, dec_mask, ga_sigma)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            ep_loss += loss.item()
        avg_train = ep_loss / max(len(train_loader), 1)
        train_losses.append(avg_train)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = batch
                text_pad = text_pad.to(device)
                mel_pad  = mel_pad.to(device)
                gate_pad = gate_pad.to(device)
                enc_mask = enc_mask.to(device)
                dec_mask = dec_mask.to(device)
                mel_out, mel_post, stop_tok, _ = model(
                    text_pad, in_lens, mel_pad, enc_mask, dec_mask
                )
                v_loss += (
                    F.mse_loss(mel_out, mel_pad)
                    + F.mse_loss(mel_post, mel_pad)
                    + F.binary_cross_entropy_with_logits(
                        stop_tok.reshape(-1, 1), gate_pad.reshape(-1, 1)
                    )
                ).item()
        avg_val = v_loss / max(len(val_loader), 1)
        val_losses.append(avg_val)
        print(f"[{speaker}] Epoch {epoch:3d}/{epochs} -- train {avg_train:.4f}  val {avg_val:.4f}  ga_w {ga_w:.3f}")

        if epoch % save_every == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                p_text, p_lens, p_mel, p_gate, p_enc_mask, p_dec_mask = probe_batch
                p_text     = p_text.to(device)
                p_mel      = p_mel.to(device)
                p_enc_mask = p_enc_mask.to(device)
                p_dec_mask = p_dec_mask.to(device)
                _, p_mel_post, _, p_attn = model(p_text, p_lens, p_mel, p_enc_mask, p_dec_mask)

            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt

            attn_np     = p_attn[0].cpu().float().numpy()
            mel_true_np = p_mel[0].cpu().float().numpy()
            mel_pred_np = p_mel_post[0].cpu().float().numpy()

            fig, axes = plt.subplots(3, 1, figsize=(10, 9))
            im0 = axes[0].imshow(attn_np, aspect="auto", origin="lower", interpolation="nearest")
            axes[0].set_title(f"Speaker {speaker} — epoch {epoch}  (ga_w={ga_w:.3f})")
            axes[0].set_xlabel("Encoder step"); axes[0].set_ylabel("Decoder step")
            fig.colorbar(im0, ax=axes[0])
            im1 = axes[1].imshow(mel_true_np.T, aspect="auto", origin="lower", interpolation="nearest")
            axes[1].set_title("Ground-truth mel"); axes[1].set_xlabel("Frame"); axes[1].set_ylabel("Mel bin")
            fig.colorbar(im1, ax=axes[1])
            vmin, vmax = mel_true_np.min(), mel_true_np.max()
            im2 = axes[2].imshow(mel_pred_np.T, aspect="auto", origin="lower", interpolation="nearest", vmin=vmin, vmax=vmax)
            axes[2].set_title("Predicted mel (post-net)"); axes[2].set_xlabel("Frame"); axes[2].set_ylabel("Mel bin")
            fig.colorbar(im2, ax=axes[2])
            plt.tight_layout()
            plot_path = align_dir / f"epoch_{epoch:04d}.png"
            fig.savefig(plot_path, dpi=100)
            plt.close(fig)
            print(f"[{speaker}]   Saved alignment plot → {plot_path.name}")

        is_milestone = (epoch % save_every == 0) or (epoch == epochs)
        ckpt_name = (
            f"speaker_{speaker}_epoch_{epoch:04d}.pth"
            if is_milestone
            else "speaker_{}_last.pth".format(speaker)
        )
        new_path = ckpt_dir / ckpt_name
        torch.save({
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch"               : epoch,
            "epochs_total"        : epochs,
            "speaker"             : speaker,
            "train_losses"        : train_losses,
            "val_losses"          : val_losses,
            "pretrained_from"     : cfg_dict["pretrained_ckpt"],
            "config"              : config,
        }, new_path)

        if keep_last and last_ckpt_path and last_ckpt_path.exists():
            stem = last_ckpt_path.stem
            if "_epoch_" in stem:
                prev_ep = int(stem.split("_epoch_")[-1])
                if prev_ep % save_every != 0:
                    last_ckpt_path.unlink(missing_ok=True)
            else:
                last_ckpt_path.unlink(missing_ok=True)

        last_ckpt_path = new_path
        print(f"[{speaker}]   Saved {new_path.name}")

    final_alias = ckpt_dir / f"speaker_{speaker}_last.pth"
    if last_ckpt_path and last_ckpt_path != final_alias:
        shutil.copy2(last_ckpt_path, final_alias)

    print(f"[{speaker}] Done. Final checkpoint: {final_alias}")
    return speaker, train_losses, val_losses


print("Worker function defined.")

## 4. Launch parallel fine-tuning

In [ ]:
import multiprocessing as mp
import concurrent.futures
import traceback

cfg = dict(
    root                    = str(ROOT),
    pretrained_ckpt         = PRETRAINED_CHECKPOINT,
    dataset_root            = DATASET_ROOT,
    ckpt_base_dir           = CKPT_BASE_DIR,
    epochs                  = FINETUNE_EPOCHS,
    batch_size              = BATCH_SIZE,
    lr                      = LEARNING_RATE,
    grad_clip               = GRAD_CLIP,
    val_split               = VAL_SPLIT,
    val_seed                = VAL_SPLIT_SEED,
    keep_last_only          = KEEP_LAST_ONLY,
    save_every              = SAVE_EVERY,
    device                  = DEVICE,
    guided_attn_weight      = GUIDED_ATTN_WEIGHT,
    guided_attn_sigma       = GUIDED_ATTN_SIGMA,
    guided_attn_anneal_start= GUIDED_ATTN_ANNEAL_START,
    guided_attn_anneal_end  = GUIDED_ATTN_ANNEAL_END,
)

results = {}

if MAX_PARALLEL == 1 or len(SPEAKERS) == 1:
    print("Running sequentially (MAX_PARALLEL=1 or single speaker).")
    for sp in SPEAKERS:
        try:
            sp_out, tl, vl = finetune_speaker(sp, cfg)
            results[sp_out] = (tl, vl)
        except Exception:
            print(f"[{sp}] FAILED:")
            traceback.print_exc()
else:
    print(f"Launching {len(SPEAKERS)} workers (max parallel={MAX_PARALLEL}) ...")
    ctx = mp.get_context("spawn")
    with concurrent.futures.ProcessPoolExecutor(
        max_workers=MAX_PARALLEL, mp_context=ctx
    ) as pool:
        futures = {pool.submit(finetune_speaker, sp, cfg): sp for sp in SPEAKERS}
        for fut in concurrent.futures.as_completed(futures):
            sp = futures[fut]
            try:
                sp_out, tl, vl = fut.result()
                results[sp_out] = (tl, vl)
                print(f"[{sp}] Finished successfully.")
            except Exception:
                print(f"[{sp}] FAILED:")
                traceback.print_exc()

print("\nAll speakers done:", list(results.keys()))

## 5. Loss curves per speaker

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not results:
    print("No results to plot — run section 4 first.")
else:
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
    for ax, (sp, (tl, vl)) in zip(axes[0], sorted(results.items())):
        ep = np.arange(1, len(tl) + 1)
        ax.plot(ep, tl, "b-o", markersize=3, label="train")
        if vl:
            ax.plot(ep, vl, "r-s", markersize=3, label="val")
        ax.set_title(f"Speaker {sp}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)
    fig.suptitle("Omani speaker fine-tuning losses", y=1.02)
    plt.tight_layout()
    plt.show()

## 5a. Visualize alignment, actual mel, and predicted mel (single batch)

In [ ]:
"""
Run one forward pass on the first validation batch for each speaker and
display three panels side-by-side:
  Left   — attention alignment  (decoder step × encoder step)
  Centre — actual mel spectrogram  (ground truth)
  Right  — predicted mel spectrogram  (post-net output)

Requires the speaker checkpoints to already exist (run section 4 first).
"""
import torch, os
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, random_split

from commons.dataset     import NewOmaniTTSDataset, TTSCollator
from commons.hyperparams import Tacotron2Config
from tacotron2.model     import Tacotron2
from tacotron2.tokenizer import Tokenizer


def _viz_speaker(speaker, ckpt_path, device_str="auto"):
    device = torch.device(
        ("cuda" if torch.cuda.is_available() else "cpu")
        if device_str == "auto" else device_str
    )

    config = Tacotron2Config()
    tok    = Tokenizer()
    config.num_chars    = tok.vocab_size
    config.pad_token_id = tok.pad_token_id

    full_ds = NewOmaniTTSDataset(
        DATASET_ROOT,
        sample_rate=config.sample_rate, n_fft=config.n_fft,
        window_size=config.win_length, hop_size=config.hop_length,
        fmin=config.fmin, fmax=config.fmax, num_mels=config.n_mels,
        min_db=config.min_db, max_scaled_abs=config.max_scaled_abs,
    )
    n_val   = max(1, int(len(full_ds) * VAL_SPLIT))
    n_train = len(full_ds) - n_val
    gen     = torch.Generator().manual_seed(VAL_SPLIT_SEED)
    _, val_ds = random_split(full_ds, [n_train, n_val], generator=gen)

    loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        collate_fn=TTSCollator(), num_workers=0)

    model = Tacotron2(config).to(device)
    ck    = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    model.eval()
    epoch = ck.get("epoch", "?")

    # grab first validation sample
    batch = next(iter(loader))
    text_pad, in_lens, mel_pad, gate_pad, enc_mask, dec_mask = batch
    text_pad = text_pad.to(device)
    mel_pad  = mel_pad.to(device)
    gate_pad = gate_pad.to(device)
    enc_mask = enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    with torch.no_grad():
        mel_out, mel_post, stop_tok, attn = model(
            text_pad, in_lens, mel_pad, enc_mask, dec_mask
        )

    # shapes → numpy  (take item 0 in batch)
    attn_np  = attn[0].cpu().float().numpy()        # (T_dec, T_enc)
    mel_true = mel_pad[0].cpu().float().numpy()     # (n_mels, T_dec)
    mel_pred = mel_post[0].cpu().float().numpy()    # (n_mels, T_dec)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # --- alignment ---
    ax = axes[0]
    im = ax.imshow(attn_np, aspect="auto", origin="lower", interpolation="nearest")
    ax.set_title(f"Speaker {speaker} — alignment (epoch {epoch})")
    ax.set_xlabel("Encoder step")
    ax.set_ylabel("Decoder step")
    fig.colorbar(im, ax=ax)

    # --- actual mel ---
    ax = axes[1]
    im = ax.imshow(mel_true, aspect="auto", origin="lower",
                   interpolation="nearest", vmin=mel_true.min(), vmax=mel_true.max())
    ax.set_title(f"Speaker {speaker} — actual mel")
    ax.set_xlabel("Time frame")
    ax.set_ylabel("Mel bin")
    fig.colorbar(im, ax=ax)

    # --- predicted mel ---
    ax = axes[2]
    vmin, vmax = mel_true.min(), mel_true.max()   # same scale as actual for fair comparison
    im = ax.imshow(mel_pred, aspect="auto", origin="lower",
                   interpolation="nearest", vmin=vmin, vmax=vmax)
    ax.set_title(f"Speaker {speaker} — predicted mel (post-net)")
    ax.set_xlabel("Time frame")
    ax.set_ylabel("Mel bin")
    fig.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

    # print per-sample loss for context
    val_loss = (
        F.mse_loss(mel_post, mel_pad)
        + F.mse_loss(mel_out, mel_pad)
        + F.binary_cross_entropy_with_logits(
            stop_tok.reshape(-1, 1), gate_pad.reshape(-1, 1)
        )
    ).item()
    print(f"[{speaker}] sample loss: {val_loss:.4f}")


# ── run for every trained speaker ─────────────────────────────────────────────
base = Path(CKPT_BASE_DIR)
for sp in sorted(SPEAKERS):
    alias = base / sp / f"speaker_{sp}_last.pth"
    if alias.exists():
        print(f"\n=== Speaker {sp} ===")
        _viz_speaker(sp, alias, device_str=DEVICE)
    else:
        print(f"[{sp}] checkpoint not found at {alias} — skipping.")

## 6. Checkpoint summary

In [ ]:
from pathlib import Path
import torch

base = Path(CKPT_BASE_DIR)
print("  Speaker  Checkpoint                              Epoch  Val loss")
print("-" * 70)
for sp in sorted(SPEAKERS):
    sp_dir = base / sp
    alias  = sp_dir / f"speaker_{sp}_last.pth"
    if alias.exists():
        ck = torch.load(alias, map_location="cpu", weights_only=False)
        vl = ck.get("val_losses", [])
        final_val = f"{vl[-1]:.4f}" if vl else "n/a"
        ep = ck.get("epoch", "?")
        print(f"  {sp:>6}  {alias.name:40}  {str(ep):>5}  {final_val:>8}")
    else:
        print(f"  {sp:>6}  (not found)")

## 7. Resume a speaker (optional)

If fine-tuning was interrupted, set `RESUME_SPEAKER` and run this cell
to continue from where it stopped.

In [ ]:
RESUME_SPEAKER      = "B"
RESUME_EXTRA_EPOCHS = 20
RESUME_CHECKPOINT   = str(
    Path(CKPT_BASE_DIR) / RESUME_SPEAKER / f"speaker_{RESUME_SPEAKER}_last.pth"
)

if not os.path.isfile(RESUME_CHECKPOINT):
    print(f"Checkpoint not found: {RESUME_CHECKPOINT} -- nothing to resume.")
else:
    ck = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=False)
    done_epochs = int(ck.get("epoch", 0))
    print(
        f"Resuming speaker {RESUME_SPEAKER} from epoch {done_epochs}, "
        f"training {RESUME_EXTRA_EPOCHS} more epochs."
    )
    resume_cfg = dict(
        root            = str(ROOT),
        pretrained_ckpt = RESUME_CHECKPOINT,
        dataset_root    = DATASET_ROOT,
        ckpt_base_dir   = CKPT_BASE_DIR,
        epochs          = RESUME_EXTRA_EPOCHS,
        batch_size      = BATCH_SIZE,
        lr              = LEARNING_RATE,
        grad_clip       = GRAD_CLIP,
        val_split       = VAL_SPLIT,
        val_seed        = VAL_SPLIT_SEED,
        keep_last_only  = KEEP_LAST_ONLY,
        save_every      = SAVE_EVERY,
        device          = DEVICE,
    )
    sp_out, tl, vl = finetune_speaker(RESUME_SPEAKER, resume_cfg)
    results[sp_out] = (tl, vl)
    print(f"Resume done. Final val loss: {vl[-1]:.4f}")